# Exploratory Data Analysis: TinyStories Dataset

## Knowledge Distillation for Lightweight Generative Language Models

This notebook provides a comprehensive exploration of the TinyStories dataset, which serves as the foundation for our knowledge distillation experiments. We investigate:

- Dataset structure and basic statistics
- Text length distributions (character-level and token-level)
- Vocabulary and word frequency analysis
- Sentence and narrative structure
- Tokenizer behavior and coverage
- Train/Validation split comparison
- Implications for model architecture decisions

In [ ]:
# Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer
import warnings
import re

warnings.filterwarnings('ignore')

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print("Libraries loaded successfully!")

## 1. Load the Dataset

We load TinyStories from the local saved copy (downloaded via `scripts/download_data.py`). The dataset contains synthetically generated children's stories designed to train small language models.

In [ ]:
# Load dataset from local disk (already downloaded)
from pathlib import Path

data_dir = Path("../data")

if (data_dir / "train").exists():
    print("Loading from local disk...")
    train_dataset = load_from_disk(str(data_dir / "train"))
    val_dataset = load_from_disk(str(data_dir / "validation"))
else:
    print("Loading from Hugging Face Hub...")
    dataset = load_dataset("roneneldan/TinyStories")
    train_dataset = dataset["train"]
    val_dataset = dataset["validation"]

print(f"Train samples: {len(train_dataset):,}")
print(f"Validation samples: {len(val_dataset):,}")
print(f"Total samples: {len(train_dataset) + len(val_dataset):,}")
print(f"\nColumns: {train_dataset.column_names}")
print(f"Features: {train_dataset.features}")

## 2. Initial Data Inspection

Let's look at a few sample stories to understand the content and structure of the data.

In [ ]:
# Display sample stories
print("=" * 80)
print("SAMPLE STORIES FROM TRAINING SET")
print("=" * 80)

for i in range(5):
    print(f"\n{'─' * 80}")
    print(f"Story #{i+1} (length: {len(train_dataset[i]['text'])} chars)")
    print(f"{'─' * 80}")
    print(train_dataset[i]["text"][:600])
    if len(train_dataset[i]["text"]) > 600:
        print("...")

## 3. Text Length Distribution (Character Level)

Understanding the length distribution is critical for choosing the `max_length` parameter for tokenization and model context windows.

In [ ]:
# Compute character lengths for a representative sample (full train set)
# Using a sample for speed in EDA
sample_size = 50000
np.random.seed(42)
sample_indices = np.random.choice(len(train_dataset), size=sample_size, replace=False)
train_sample_texts = [train_dataset[int(i)]["text"] for i in sample_indices]
val_texts = [val_dataset[i]["text"] for i in range(len(val_dataset))]

# Character lengths
train_char_lengths = [len(t) for t in train_sample_texts]
val_char_lengths = [len(t) for t in val_texts]

# Statistics
print("=== Character Length Statistics ===")
print(f"\nTrain (sample of {sample_size:,}):")
print(f"  Mean: {np.mean(train_char_lengths):.0f}")
print(f"  Median: {np.median(train_char_lengths):.0f}")
print(f"  Std: {np.std(train_char_lengths):.0f}")
print(f"  Min: {np.min(train_char_lengths)}")
print(f"  Max: {np.max(train_char_lengths)}")
print(f"  25th percentile: {np.percentile(train_char_lengths, 25):.0f}")
print(f"  75th percentile: {np.percentile(train_char_lengths, 75):.0f}")
print(f"  95th percentile: {np.percentile(train_char_lengths, 95):.0f}")
print(f"  99th percentile: {np.percentile(train_char_lengths, 99):.0f}")

print(f"\nValidation ({len(val_texts):,} samples):")
print(f"  Mean: {np.mean(val_char_lengths):.0f}")
print(f"  Median: {np.median(val_char_lengths):.0f}")
print(f"  Std: {np.std(val_char_lengths):.0f}")

In [ ]:
# Plot character length distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram
axes[0].hist(train_char_lengths, bins=100, alpha=0.7, color='steelblue', edgecolor='white', label='Train')
axes[0].hist(val_char_lengths, bins=100, alpha=0.5, color='coral', edgecolor='white', label='Validation')
axes[0].axvline(np.median(train_char_lengths), color='navy', linestyle='--', linewidth=2, label=f'Median: {np.median(train_char_lengths):.0f}')
axes[0].axvline(np.percentile(train_char_lengths, 95), color='red', linestyle=':', linewidth=2, label=f'95th pct: {np.percentile(train_char_lengths, 95):.0f}')
axes[0].set_xlabel('Character Length')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Story Lengths (Characters)')
axes[0].legend()
axes[0].set_xlim(0, 3000)

# Box plot comparison
data_for_box = [train_char_lengths, val_char_lengths]
bp = axes[1].boxplot(data_for_box, labels=['Train', 'Validation'], patch_artist=True)
bp['boxes'][0].set_facecolor('steelblue')
bp['boxes'][1].set_facecolor('coral')
axes[1].set_ylabel('Character Length')
axes[1].set_title('Character Length: Train vs Validation')

plt.tight_layout()
plt.savefig('../reports/char_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nPlot saved to reports/char_length_distribution.png")

## 4. Token Length Distribution (GPT-2 Tokenizer)

Since our models operate on tokens (not characters), we need to understand the token-level length distribution. This directly informs our `max_length` parameter choice.

In [ ]:
# Load GPT-2 tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
print(f"Tokenizer: GPT-2 BPE")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")
print(f"Model max length: {tokenizer.model_max_length:,}")

# Tokenize sample texts
print("\nTokenizing sample texts (this may take a moment)...")
train_token_lengths = []
for text in train_sample_texts:
    tokens = tokenizer.encode(text)
    train_token_lengths.append(len(tokens))

val_token_lengths = []
for text in val_texts:
    tokens = tokenizer.encode(text)
    val_token_lengths.append(len(tokens))

print(f"\n=== Token Length Statistics ===")
print(f"\nTrain (sample of {sample_size:,}):")
print(f"  Mean: {np.mean(train_token_lengths):.0f} tokens")
print(f"  Median: {np.median(train_token_lengths):.0f} tokens")
print(f"  Std: {np.std(train_token_lengths):.0f}")
print(f"  Min: {np.min(train_token_lengths)} tokens")
print(f"  Max: {np.max(train_token_lengths)} tokens")
print(f"  25th percentile: {np.percentile(train_token_lengths, 25):.0f}")
print(f"  75th percentile: {np.percentile(train_token_lengths, 75):.0f}")
print(f"  90th percentile: {np.percentile(train_token_lengths, 90):.0f}")
print(f"  95th percentile: {np.percentile(train_token_lengths, 95):.0f}")
print(f"  99th percentile: {np.percentile(train_token_lengths, 99):.0f}")

# Context window coverage
for ctx_len in [128, 256, 512, 1024]:
    coverage = np.mean(np.array(train_token_lengths) <= ctx_len) * 100
    print(f"\n  Stories fitting in {ctx_len} tokens: {coverage:.1f}%")

In [ ]:
# Plot token length distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram with context window markers
axes[0].hist(train_token_lengths, bins=100, alpha=0.7, color='steelblue', edgecolor='white')
axes[0].axvline(256, color='orange', linestyle='--', linewidth=2, label='256 tokens (student ctx)')
axes[0].axvline(512, color='red', linestyle='--', linewidth=2, label='512 tokens (chosen max_length)')
axes[0].axvline(1024, color='darkred', linestyle=':', linewidth=2, label='1024 tokens (GPT-2 max)')
axes[0].axvline(np.median(train_token_lengths), color='navy', linestyle='-', linewidth=2, 
                label=f'Median: {np.median(train_token_lengths):.0f}')
axes[0].set_xlabel('Number of Tokens')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Token Length Distribution (GPT-2 Tokenizer)')
axes[0].legend(fontsize=9)
axes[0].set_xlim(0, 1200)

# Cumulative distribution
sorted_lengths = np.sort(train_token_lengths)
cumulative = np.arange(1, len(sorted_lengths) + 1) / len(sorted_lengths) * 100
axes[1].plot(sorted_lengths, cumulative, color='steelblue', linewidth=2)
axes[1].axhline(90, color='gray', linestyle=':', alpha=0.5)
axes[1].axhline(95, color='gray', linestyle=':', alpha=0.5)
axes[1].axvline(256, color='orange', linestyle='--', linewidth=1.5, label='256 tokens')
axes[1].axvline(512, color='red', linestyle='--', linewidth=1.5, label='512 tokens')
axes[1].set_xlabel('Number of Tokens')
axes[1].set_ylabel('Cumulative % of Stories')
axes[1].set_title('Cumulative Token Length Distribution')
axes[1].legend()
axes[1].set_xlim(0, 1200)
axes[1].set_ylim(0, 100)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/token_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nPlot saved to reports/token_length_distribution.png")

## 5. Word Frequency Analysis

Analyzing the most common words reveals the vocabulary complexity and thematic patterns of TinyStories.

In [ ]:
# Word frequency analysis
all_words = []
for text in train_sample_texts[:20000]:  # Use 20k samples for word analysis
    words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
    all_words.extend(words)

word_counts = Counter(all_words)
total_words = len(all_words)
unique_words = len(word_counts)

print(f"Total words analyzed: {total_words:,}")
print(f"Unique words: {unique_words:,}")
print(f"Vocabulary richness (unique/total): {unique_words/total_words:.4f}")

# Top 30 most common words
print(f"\n{'─' * 50}")
print("Top 30 Most Frequent Words:")
print(f"{'─' * 50}")
for word, count in word_counts.most_common(30):
    pct = count / total_words * 100
    print(f"  {word:15s} {count:>8,} ({pct:.2f}%)")

In [ ]:
# Plot word frequencies
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Top 25 words bar chart
top_words = word_counts.most_common(25)
words, counts = zip(*top_words)
axes[0].barh(range(len(words)), counts, color='steelblue', edgecolor='white')
axes[0].set_yticks(range(len(words)))
axes[0].set_yticklabels(words)
axes[0].invert_yaxis()
axes[0].set_xlabel('Frequency')
axes[0].set_title('Top 25 Most Common Words')

# Word frequency distribution (Zipf's law)
freqs = sorted(word_counts.values(), reverse=True)
axes[1].loglog(range(1, len(freqs) + 1), freqs, color='steelblue', alpha=0.7, linewidth=0.5)
axes[1].set_xlabel('Word Rank (log scale)')
axes[1].set_ylabel('Frequency (log scale)')
axes[1].set_title("Word Frequency Distribution (Zipf's Law)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/word_frequency_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Content-Specific Word Analysis

Let's look at words that are NOT common stop words to understand the thematic content of the stories (characters, objects, actions, emotions).

In [ ]:
# Filter out common stop words to see content words
stop_words = {
    'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
    'of', 'with', 'by', 'from', 'is', 'was', 'were', 'are', 'be', 'been',
    'being', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would',
    'could', 'should', 'may', 'might', 'shall', 'can', 'need', 'dare',
    'it', 'its', 'he', 'she', 'they', 'we', 'i', 'you', 'me', 'him',
    'her', 'us', 'them', 'my', 'your', 'his', 'their', 'our', 'this',
    'that', 'these', 'those', 'what', 'which', 'who', 'whom', 'where',
    'when', 'why', 'how', 'not', 'no', 'so', 'very', 'too', 'also',
    'just', 'then', 'than', 'if', 'as', 'up', 'out', 'about', 'into',
    'over', 'after', 'before', 'between', 'under', 'again', 'there',
    'here', 'all', 'each', 'every', 'both', 'few', 'more', 'most',
    'other', 'some', 'such', 'only', 'own', 'same', 's', 't', 'said'
}

content_words = {word: count for word, count in word_counts.items() 
                 if word not in stop_words and len(word) > 2}
content_counter = Counter(content_words)

# Categorize top content words
print("=== Top Content Words (excluding stop words) ===\n")

# Names
names = [w for w, c in content_counter.most_common(200) if w[0].islower() == False or w in 
         ['lily', 'timmy', 'tom', 'sam', 'max', 'ben', 'lucy', 'anna', 'jack', 'mia', 'emma']]

print("Top 30 Content Words:")
for word, count in content_counter.most_common(30):
    print(f"  {word:15s} {count:>7,}")

# Plot content words
fig, ax = plt.subplots(figsize=(14, 8))
top_content = content_counter.most_common(40)
words_c, counts_c = zip(*top_content)
colors = ['#e74c3c' if w in ['happy', 'sad', 'scared', 'excited', 'angry', 'proud', 'surprised', 'lonely'] 
          else '#3498db' if w in ['play', 'went', 'came', 'saw', 'got', 'wanted', 'looked', 'told', 'asked', 'knew', 'started', 'found', 'made', 'liked', 'ran']
          else '#2ecc71' for w in words_c]

ax.barh(range(len(words_c)), counts_c, color=colors, edgecolor='white')
ax.set_yticks(range(len(words_c)))
ax.set_yticklabels(words_c, fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('Frequency')
ax.set_title('Top 40 Content Words\n(Red=Emotions, Blue=Actions, Green=Other)')

plt.tight_layout()
plt.savefig('../reports/content_words.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Sentence Structure Analysis

Understanding the sentence-level structure helps us gauge grammatical complexity and narrative patterns.

In [ ]:
# Sentence structure analysis
sentences_per_story = []
words_per_sentence = []
words_per_story = []

for text in train_sample_texts[:20000]:
    # Split into sentences (rough heuristic)
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    sentences_per_story.append(len(sentences))
    
    # Words per story
    story_words = text.split()
    words_per_story.append(len(story_words))
    
    # Words per sentence
    for sent in sentences:
        word_count = len(sent.split())
        if word_count > 0:
            words_per_sentence.append(word_count)

print("=== Sentence Structure Statistics ===")
print(f"\nSentences per story:")
print(f"  Mean: {np.mean(sentences_per_story):.1f}")
print(f"  Median: {np.median(sentences_per_story):.0f}")
print(f"  Min: {np.min(sentences_per_story)}")
print(f"  Max: {np.max(sentences_per_story)}")

print(f"\nWords per sentence:")
print(f"  Mean: {np.mean(words_per_sentence):.1f}")
print(f"  Median: {np.median(words_per_sentence):.0f}")
print(f"  Min: {np.min(words_per_sentence)}")
print(f"  Max: {np.max(words_per_sentence)}")

print(f"\nWords per story:")
print(f"  Mean: {np.mean(words_per_story):.0f}")
print(f"  Median: {np.median(words_per_story):.0f}")

In [ ]:
# Plot sentence structure
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sentences per story
axes[0, 0].hist(sentences_per_story, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0, 0].axvline(np.median(sentences_per_story), color='red', linestyle='--', 
                   label=f'Median: {np.median(sentences_per_story):.0f}')
axes[0, 0].set_xlabel('Number of Sentences')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Sentences per Story')
axes[0, 0].legend()

# Words per sentence
axes[0, 1].hist(words_per_sentence, bins=50, color='coral', edgecolor='white', alpha=0.8, range=(0, 50))
axes[0, 1].axvline(np.median(words_per_sentence), color='navy', linestyle='--',
                   label=f'Median: {np.median(words_per_sentence):.0f}')
axes[0, 1].set_xlabel('Words per Sentence')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Words per Sentence Distribution')
axes[0, 1].legend()

# Words per story
axes[1, 0].hist(words_per_story, bins=80, color='seagreen', edgecolor='white', alpha=0.8)
axes[1, 0].axvline(np.median(words_per_story), color='red', linestyle='--',
                   label=f'Median: {np.median(words_per_story):.0f}')
axes[1, 0].set_xlabel('Words per Story')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Words per Story Distribution')
axes[1, 0].legend()

# Character-to-token ratio
char_to_token = np.array(train_char_lengths[:20000]) / np.array(train_token_lengths[:20000])
axes[1, 1].hist(char_to_token, bins=50, color='mediumpurple', edgecolor='white', alpha=0.8)
axes[1, 1].axvline(np.median(char_to_token), color='red', linestyle='--',
                   label=f'Median: {np.median(char_to_token):.2f} chars/token')
axes[1, 1].set_xlabel('Characters per Token')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Character-to-Token Ratio')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('../reports/sentence_structure.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Vocabulary Coverage Analysis

How much of the GPT-2 vocabulary is actually used by TinyStories? This tells us about the effective complexity of the dataset.

In [ ]:
# Vocabulary coverage analysis
token_counter = Counter()
for text in train_sample_texts[:20000]:
    tokens = tokenizer.encode(text)
    token_counter.update(tokens)

used_tokens = len(token_counter)
total_vocab = tokenizer.vocab_size

print("=== Vocabulary Coverage ===")
print(f"GPT-2 vocabulary size: {total_vocab:,}")
print(f"Tokens used in TinyStories sample: {used_tokens:,}")
print(f"Vocabulary utilization: {used_tokens/total_vocab*100:.1f}%")
print(f"Unused tokens: {total_vocab - used_tokens:,}")

# Token frequency distribution
token_freqs = sorted(token_counter.values(), reverse=True)
print(f"\nMost frequent tokens cover:")
cumsum = np.cumsum(token_freqs) / sum(token_freqs) * 100
for threshold in [50, 80, 90, 95, 99]:
    n_tokens = np.searchsorted(cumsum, threshold) + 1
    print(f"  {threshold}% of text: {n_tokens:,} tokens ({n_tokens/total_vocab*100:.1f}% of vocab)")

# Most common tokens
print(f"\nTop 20 most common tokens:")
for token_id, count in token_counter.most_common(20):
    token_str = tokenizer.decode([token_id])
    print(f"  {repr(token_str):20s} (id={token_id:5d}): {count:>8,}")

In [ ]:
# Vocabulary coverage visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Token frequency distribution (log scale)
axes[0].semilogy(range(1, len(token_freqs) + 1), token_freqs, color='steelblue', linewidth=0.8)
axes[0].set_xlabel('Token Rank')
axes[0].set_ylabel('Frequency (log scale)')
axes[0].set_title('Token Frequency Distribution')
axes[0].axvline(1000, color='red', linestyle='--', alpha=0.7, label='Top 1000 tokens')
axes[0].axvline(5000, color='orange', linestyle='--', alpha=0.7, label='Top 5000 tokens')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Cumulative coverage
cumsum_pct = np.cumsum(token_freqs) / sum(token_freqs) * 100
axes[1].plot(range(1, len(cumsum_pct) + 1), cumsum_pct, color='steelblue', linewidth=2)
axes[1].axhline(90, color='gray', linestyle=':', alpha=0.5, label='90% coverage')
axes[1].axhline(95, color='gray', linestyle='--', alpha=0.5, label='95% coverage')
axes[1].axhline(99, color='gray', linestyle='-', alpha=0.3, label='99% coverage')
axes[1].set_xlabel('Number of Unique Tokens')
axes[1].set_ylabel('Cumulative Text Coverage (%)')
axes[1].set_title('Vocabulary Coverage (Cumulative)')
axes[1].legend()
axes[1].set_xlim(0, 15000)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/vocabulary_coverage.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Story Opening Patterns

TinyStories has characteristic narrative patterns. Let's analyze how stories typically begin.

In [ ]:
# Analyze story openings
opening_patterns = Counter()
first_words = Counter()

for text in train_sample_texts[:20000]:
    # First N words
    words = text.split()[:5]
    opening = ' '.join(words)
    opening_patterns[opening] += 1
    if words:
        first_words[words[0].lower()] += 1

# Common first words
print("=== Story Opening Analysis ===")
print(f"\nMost common first words:")
for word, count in first_words.most_common(15):
    pct = count / 20000 * 100
    print(f"  {word:15s} {count:>5,} ({pct:.1f}%)")

# Common opening phrases
print(f"\nMost common opening phrases (first 5 words):")
for phrase, count in opening_patterns.most_common(15):
    print(f"  \"{phrase}...\" — {count:,} times")

# Visualize first words
fig, ax = plt.subplots(figsize=(12, 6))
top_first = first_words.most_common(15)
words_f, counts_f = zip(*top_first)
bars = ax.bar(words_f, counts_f, color='steelblue', edgecolor='white')
ax.set_xlabel('First Word')
ax.set_ylabel('Frequency')
ax.set_title('Most Common Story Opening Words')
ax.set_xticklabels(words_f, rotation=45, ha='right')

# Add percentage labels
total = 20000
for bar, count in zip(bars, counts_f):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{count/total*100:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/story_openings.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Train vs Validation Split Comparison

We verify that the training and validation sets have similar distributions.

In [ ]:
# Compare train and validation distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Character length comparison
axes[0, 0].hist(train_char_lengths, bins=80, alpha=0.6, color='steelblue', 
                density=True, label=f'Train (n={sample_size:,})', edgecolor='white')
axes[0, 0].hist(val_char_lengths, bins=80, alpha=0.6, color='coral', 
                density=True, label=f'Validation (n={len(val_texts):,})', edgecolor='white')
axes[0, 0].set_xlabel('Character Length')
axes[0, 0].set_ylabel('Density')
axes[0, 0].set_title('Character Length: Train vs Validation')
axes[0, 0].legend()
axes[0, 0].set_xlim(0, 2500)

# Token length comparison
axes[0, 1].hist(train_token_lengths, bins=80, alpha=0.6, color='steelblue', 
                density=True, label='Train', edgecolor='white')
axes[0, 1].hist(val_token_lengths, bins=80, alpha=0.6, color='coral', 
                density=True, label='Validation', edgecolor='white')
axes[0, 1].set_xlabel('Token Length')
axes[0, 1].set_ylabel('Density')
axes[0, 1].set_title('Token Length: Train vs Validation')
axes[0, 1].legend()
axes[0, 1].set_xlim(0, 800)

# Word count per story comparison
val_words_per_story = [len(text.split()) for text in val_texts[:5000]]
train_words_sample = words_per_story[:5000]
axes[1, 0].hist(train_words_sample, bins=60, alpha=0.6, color='steelblue', 
                density=True, label='Train', edgecolor='white')
axes[1, 0].hist(val_words_per_story, bins=60, alpha=0.6, color='coral', 
                density=True, label='Validation', edgecolor='white')
axes[1, 0].set_xlabel('Words per Story')
axes[1, 0].set_ylabel('Density')
axes[1, 0].set_title('Words per Story: Train vs Validation')
axes[1, 0].legend()

# Summary statistics comparison table
stats_data = {
    'Metric': ['Char Length (mean)', 'Char Length (median)', 'Token Length (mean)', 
               'Token Length (median)', 'Samples'],
    'Train': [f"{np.mean(train_char_lengths):.0f}", f"{np.median(train_char_lengths):.0f}",
              f"{np.mean(train_token_lengths):.0f}", f"{np.median(train_token_lengths):.0f}",
              f"{len(train_dataset):,}"],
    'Validation': [f"{np.mean(val_char_lengths):.0f}", f"{np.median(val_char_lengths):.0f}",
                   f"{np.mean(val_token_lengths):.0f}", f"{np.median(val_token_lengths):.0f}",
                   f"{len(val_dataset):,}"],
}
axes[1, 1].axis('off')
table = axes[1, 1].table(
    cellText=list(zip(stats_data['Metric'], stats_data['Train'], stats_data['Validation'])),
    colLabels=['Metric', 'Train', 'Validation'],
    loc='center',
    cellLoc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)
axes[1, 1].set_title('Summary Statistics Comparison', pad=20)

plt.tight_layout()
plt.savefig('../reports/train_val_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Tokenizer Behavior Deep Dive

Let's examine how GPT-2's BPE tokenizer handles TinyStories text, including subword splitting patterns.

In [ ]:
# Tokenizer behavior analysis
print("=== Tokenizer Behavior on TinyStories ===\n")

# Example tokenization
example_texts = [
    "Once upon a time, there was a little girl named Lily.",
    "She was very happy and liked to play in the park.",
    "The big brown dog ran quickly through the beautiful garden.",
    "Timmy felt sad because his favorite toy was broken.",
]

for text in example_texts:
    tokens = tokenizer.encode(text)
    token_strs = [tokenizer.decode([t]) for t in tokens]
    print(f"Text: \"{text}\"")
    print(f"  Tokens ({len(tokens)}): {token_strs}")
    print(f"  IDs: {tokens}")
    print()

# Analyze subword patterns
print("=" * 60)
print("Subword splitting analysis:")
print("=" * 60)

# Count how many tokens are full words vs subwords
full_word_count = 0
subword_count = 0
for text in train_sample_texts[:5000]:
    tokens = tokenizer.encode(text)
    for i, t in enumerate(tokens):
        decoded = tokenizer.decode([t])
        if decoded.startswith(' ') or i == 0:
            full_word_count += 1
        else:
            subword_count += 1

total = full_word_count + subword_count
print(f"\nFull-word tokens: {full_word_count:,} ({full_word_count/total*100:.1f}%)")
print(f"Subword tokens: {subword_count:,} ({subword_count/total*100:.1f}%)")
print(f"→ Most words in TinyStories are tokenized as single tokens (simple vocabulary)")
print(f"   This is expected: children's stories use common, simple words.")

## 12. Data Quality Check

Check for potential issues: empty stories, duplicates, encoding problems, or extremely short/long outliers.

In [ ]:
# Data quality checks
print("=== Data Quality Analysis ===\n")

# Empty or very short stories
empty_count = sum(1 for t in train_sample_texts if len(t.strip()) == 0)
very_short = sum(1 for t in train_sample_texts if 0 < len(t.strip()) < 50)
very_long = sum(1 for t in train_sample_texts if len(t) > 3000)

print(f"Empty stories: {empty_count} ({empty_count/sample_size*100:.3f}%)")
print(f"Very short stories (<50 chars): {very_short} ({very_short/sample_size*100:.3f}%)")
print(f"Very long stories (>3000 chars): {very_long} ({very_long/sample_size*100:.3f}%)")

# Check for non-ASCII characters
non_ascii_count = sum(1 for t in train_sample_texts if any(ord(c) > 127 for c in t))
print(f"Stories with non-ASCII chars: {non_ascii_count} ({non_ascii_count/sample_size*100:.2f}%)")

# Check for potential duplicates (first 100 chars)
openings = [t[:100] for t in train_sample_texts]
unique_openings = len(set(openings))
potential_dupes = sample_size - unique_openings
print(f"Potential duplicates (same first 100 chars): {potential_dupes} ({potential_dupes/sample_size*100:.2f}%)")

# Show some edge cases
print(f"\n--- Shortest stories ---")
sorted_by_len = sorted(enumerate(train_sample_texts), key=lambda x: len(x[1]))
for idx, text in sorted_by_len[:3]:
    if text.strip():
        print(f"  [{len(text)} chars]: \"{text[:200]}\"")

print(f"\n--- Longest stories (truncated) ---")
for idx, text in sorted_by_len[-3:]:
    print(f"  [{len(text)} chars]: \"{text[:150]}...\"")

## 13. Model Architecture Comparison

Visual comparison of the teacher (GPT-2 Small) vs student model to understand the compression ratio.

In [ ]:
# Model architecture comparison
import sys
sys.path.insert(0, '..')
from src.models.teacher import TeacherModel
from src.models.student import StudentModel

teacher = TeacherModel(pretrained=False)
student = StudentModel(vocab_size=50257, n_layers=4, n_heads=4, hidden_size=256, max_length=512)

teacher_params = teacher.num_parameters
student_params = student.num_parameters

print("=" * 60)
print("MODEL ARCHITECTURE COMPARISON")
print("=" * 60)

comparison = {
    'Property': ['Parameters', 'Layers', 'Hidden Size', 'Attention Heads', 
                 'Context Length', 'Size (FP32)', 'Size (FP16)', 'Compression Ratio'],
    'Teacher (GPT-2)': [f'{teacher_params:,}', '12', '768', '12', 
                        '1024', f'{teacher_params*4/1024**2:.0f} MB', 
                        f'{teacher_params*2/1024**2:.0f} MB', '1.0x (baseline)'],
    'Student': [f'{student_params:,}', '4', '256', '4',
                '512', f'{student_params*4/1024**2:.0f} MB',
                f'{student_params*2/1024**2:.0f} MB', f'{teacher_params/student_params:.1f}x'],
}

df_comparison = pd.DataFrame(comparison)
print(df_comparison.to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Parameter count comparison
models = ['Teacher\n(GPT-2 Small)', 'Student\n(4-layer)']
params = [teacher_params / 1e6, student_params / 1e6]
colors = ['#e74c3c', '#3498db']
bars = axes[0].bar(models, params, color=colors, edgecolor='white', width=0.5)
axes[0].set_ylabel('Parameters (Millions)')
axes[0].set_title('Parameter Count')
for bar, p in zip(bars, params):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{p:.1f}M', ha='center', fontsize=12, fontweight='bold')

# Memory comparison (FP32, FP16, INT8)
precisions = ['FP32', 'FP16', 'INT8']
teacher_sizes = [teacher_params * 4 / 1024**2, teacher_params * 2 / 1024**2, teacher_params / 1024**2]
student_sizes = [student_params * 4 / 1024**2, student_params * 2 / 1024**2, student_params / 1024**2]

x = np.arange(len(precisions))
width = 0.35
axes[1].bar(x - width/2, teacher_sizes, width, label='Teacher', color='#e74c3c', edgecolor='white')
axes[1].bar(x + width/2, student_sizes, width, label='Student', color='#3498db', edgecolor='white')
axes[1].set_ylabel('Model Size (MB)')
axes[1].set_title('Memory Footprint by Precision')
axes[1].set_xticks(x)
axes[1].set_xticklabels(precisions)
axes[1].legend()

# Layer-by-layer parameter distribution (student)
layer_names = ['Embeddings', 'Layer 1', 'Layer 2', 'Layer 3', 'Layer 4', 'LM Head']
# Approximate parameter distribution for student
embed_params = 50257 * 256 + 512 * 256  # token + position embeddings
layer_params = (4 * 256 * 256 * 3 + 256 * 256 + 256 * 1024 * 2 + 256 * 4)  # approx per layer
head_params = 50257 * 256
layer_sizes = [embed_params/1e6, layer_params/1e6, layer_params/1e6, layer_params/1e6, layer_params/1e6, head_params/1e6]

axes[2].pie(layer_sizes, labels=layer_names, autopct='%1.1f%%', colors=sns.color_palette("husl", 6))
axes[2].set_title('Student Model Parameter Distribution')

plt.tight_layout()
plt.savefig('../reports/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Key Findings and Implications for Model Design

### Summary of EDA Findings:

1. **Dataset Size**: 2.1M training stories, 22K validation — large enough for meaningful training, small enough for consumer hardware
2. **Story Length**: Median ~200 tokens, 95% fit within 512 tokens → **max_length=512 is well-justified**
3. **Simple Vocabulary**: TinyStories uses only ~20-30% of GPT-2's vocabulary, with highly concentrated word frequencies (top 5000 tokens cover >95% of text)
4. **Consistent Structure**: Stories follow predictable patterns ("Once upon a time...", character + action + outcome), making them ideal for studying compression effects
5. **Clean Data**: Minimal quality issues (few empty/duplicate stories), almost no non-ASCII characters
6. **Tokenizer Efficiency**: Most words tokenize as single tokens due to simple vocabulary — the BPE tokenizer is efficient on this data
7. **Train/Validation Alignment**: Distributions are well-matched between splits

### Implications for Distillation:
- The simple vocabulary means a small student model CAN potentially learn token distributions
- Narrative structure provides multi-level language patterns (word → sentence → story level)
- The gap between a pretrained teacher and a random student will be large → clear room for distillation to help
- Context length of 512 tokens captures virtually all stories without truncation

In [ ]:
# Final summary visualization
fig, ax = plt.subplots(figsize=(14, 8))
ax.axis('off')

summary_text = """
╔══════════════════════════════════════════════════════════════════════════════════╗
║                    TinyStories Dataset — EDA Summary                            ║
╠══════════════════════════════════════════════════════════════════════════════════╣
║                                                                                  ║
║  Dataset:        TinyStories (Eldan & Li, 2023)                                 ║
║  Train Size:     2,119,719 stories                                               ║
║  Val Size:       21,990 stories                                                  ║
║  Avg Length:     ~200 tokens (median)                                            ║
║  Vocabulary:     ~20-30%% of GPT-2 vocab actually used                           ║
║  Quality:        Very clean, minimal issues                                      ║
║                                                                                  ║
║  ─────────────────────────────────────────────────────────────────────────────── ║
║                                                                                  ║
║  Teacher:        GPT-2 Small (124M params, 12 layers, 768 hidden)               ║
║  Student:        Custom (16M params, 4 layers, 256 hidden)                      ║
║  Compression:    7.7x parameter reduction                                        ║
║  Max Length:     512 tokens (covers 95%%+ of stories)                             ║
║                                                                                  ║
║  ─────────────────────────────────────────────────────────────────────────────── ║
║                                                                                  ║
║  Status:         ✓ Data downloaded and explored                                  ║
║                  ✓ Models defined and verified                                   ║
║                  ✓ Pipeline ready for training                                   ║
║                                                                                  ║
╚══════════════════════════════════════════════════════════════════════════════════╝
"""

ax.text(0.5, 0.5, summary_text, transform=ax.transAxes, fontsize=11,
        verticalalignment='center', horizontalalignment='center',
        fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='lightcyan', alpha=0.8))

plt.tight_layout()
plt.savefig('../reports/eda_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nEDA Complete! All plots saved to ../reports/")